# CS341 Submission 2 — Reproducible NYC Taxi Data Pipeline (H1 2024)

**โจทย์หลัก:** *Which taxi type was the most used in the first half of 2024?*

Notebook นี้เป็นส่วนหนึ่งของ **Submission 2 (Reproducible Data Pipeline)** ซึ่งพัฒนาต่อยอดจาก `NYC_Taxi.ipynb` (Submission 1 — Baseline) โดยเปลี่ยนจากการประมวลผลแบบ In-Memory Loop ที่ลบข้อมูลทิ้งทุกครั้งหลังนับเสร็จ (`del df`) มาเป็น **Modular & Idempotent Data Pipeline** ที่แยกชั้นการประมวลผลออกเป็น 4 ส่วน:
1. **Multi-Source Ingestion & Discovery**: รองรับการดึงไฟล์ Parquet ทั้ง 4 ประเภท (`yellow`, `green`, `fhv`, `fhvhv`) จากทั้ง Local / Google Drive / Official NYC TLC CloudFront CDN พร้อมเชื่อมกับข้อมูลอ้างอิง `taxi_zone_lookup.csv`
2. **Schema Harmonization & Strict Data Quality Gate**: แปลงโครงสร้างคอลัมน์ให้เป็นมาตรฐานเดียวกัน และตรวจสอบคุณภาพข้อมูล 5 กฎ (Null Timestamps, `dropoff < pickup`, Negative/Null Distance & Fare, Out-of-H1-2024 Date Range, และ Extreme Duration > 24 ชั่วโมง)
3. **Curated Output & State Tracking (Idempotent / Incremental)**: บันทึกชุดข้อมูลพร้อมใช้ลง `data/curated/` และบันทึกสถานะการรันลง `outputs/pipeline_state.json` ทำให้รันซ้ำได้ทันทีโดยไม่ต้องประมวลผลไฟล์เก่าซ้ำ
4. **Decoupled Downstream Analytics**: วิเคราะห์และสร้างกราฟคำตอบจาก **Curated Layer** โดยตรงภายในเวลาไม่กี่วินาที

In [ ]:
from src.pipeline import run_pipeline

# รัน Pipeline (หากเคยประมวลผล Partition ใดไปแล้ว Pipeline จะข้ามให้อัตโนมัติตาม pipeline_state.json)
state = run_pipeline(force=False, max_workers=3)
print(f"Total tracked partitions in manifest: {len(state['partitions'])}")

2026-11-15 13:43:52 | INFO    | ===========================================================================
2026-11-15 13:43:52 | INFO    | 🚀 STARTING NYC TAXI DATA PIPELINE (rule_version=2.0.0, force=False)
2026-11-15 13:43:52 | INFO    | ===========================================================================
2026-11-15 13:43:52 | INFO    | [INGEST-REF] Downloading reference CSV from https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
2026-11-15 13:43:52 | INFO    | [VALIDATED-REF] taxi_zone_lookup.csv loaded successfully (265 zones).
2026-11-15 13:43:52 | INFO    | [PLAN] Total partitions requested: 24 | To process: 24 | Skipped (cached): 0
2026-11-15 13:43:52 | INFO    | [PROCESSED] green_2024-01   | raw=    56,551 | sub1_clean=    56,369 | sub2_clean=    56,367 | dropped=   184 | time=0.37s
2026-11-15 13:43:55 | INFO    | [PROCESSED] fhv_2024-01     | raw= 1,290,116 | sub1_clean= 1,290,116 | sub2_clean= 1,289,685 | dropped=   431 | time=3.35s
2026-11-15 13:43:57 | I

## 2. ผลการตรวจสอบคุณภาพข้อมูล (Data Quality Report: Submission 1 vs Submission 2)

ใน Submission 2 เราได้ยกระดับการตรวจสอบคุณภาพข้อมูลให้ครอบคลุมยิ่งขึ้น พร้อมทั้งเก็บตัวเลข `baseline_clean_rows` ไว้เพื่อตรวจสอบย้อนกลับ (Reconcile) กับผลลัพธ์ใน Submission 1 ได้ตรงกัน 100% (`149,327,659` แถว):

In [ ]:
import json
import pandas as pd

with open("outputs/data_quality_summary.json") as f:
    dq_summary = json.load(f)

print("=== DATA QUALITY SUMMARY (H1 2024 — 24 FILES) ===")
print(json.dumps(dq_summary, indent=2, ensure_ascii=False))

dq_df = pd.read_csv("outputs/data_quality_report.csv")
dq_df[[
    "file_name", "raw_rows", "baseline_clean_rows", "strict_clean_rows",
    "dropped_negative_duration", "dropped_negative_dist_or_fare",
    "dropped_out_of_period", "dropped_extreme_duration", "retention_rate_pct"
]]

=== DATA QUALITY SUMMARY (H1 2024 — 24 FILES) ===
{
  "generated_at": "2026-11-15T06:44:35.946448+00:00",
  "rule_version": "2.0.0",
  "total_files_checked": 24,
  "all_schemas_valid": true,
  "total_raw_rows": 149655796,
  "submission1_baseline_clean_rows": 149327659,
  "submission1_baseline_dropped_rows": 328137,
  "submission2_strict_clean_rows": 149325288,
  "submission2_strict_dropped_rows": 330508,
  "additional_anomalies_caught_in_sub2": 2371,
  "anomaly_breakdown_by_reason": {
    "null_timestamps": 0,
    "negative_duration_dropoff_lt_pickup": 300,
    "negative_or_null_distance_or_fare": 327837,
    "out_of_h1_2024_date_range": 43,
    "extreme_duration_gt_24h": 2328
  }
}


                          file_name  raw_rows  baseline_clean_rows  strict_clean_rows  dropped_negative_duration  dropped_negative_dist_or_fare  dropped_out_of_period  dropped_extreme_duration  retention_rate_pct
0      fhv_tripdata_2024-01.parquet   1290116              1290116            1289685                          0                              0                      0                       431             99.9666
1    fhvhv_tripdata_2024-01.parquet  19663930             19663739           19663739                          0                            191                      0                         0             99.9990
2    green_tripdata_2024-01.parquet     56551                56369              56367                          0                            182                      2                         0             99.6746
3   yellow_tripdata_2024-01.parquet   2964624              2927120            2927091                         56                          37448     

## 3. ตัวอย่างชุดข้อมูลพร้อมใช้ (Curated Output Datasets)

ตามแนวคิดที่ได้วางไว้ตอนท้ายของ Submission 1 (`#idea การแก้ไข: ต้อง clean แล้วไปเก็บใน storage ทีละไฟล์ แล้วค่อยนำมาคำนวณใหม่จากไฟล์ที่เก็บไว้`) ตัว Pipeline ได้สร้างชุดข้อมูล Curated เก็บไว้ใน `data/curated/` ดังนี้:
1. `data/curated/curated_monthly_summary.parquet` (และ `.csv`) — ข้อมูลสรุปรายเดือนและประเภทแท็กซี่ที่ทำความสะอาดแล้ว
2. `data/curated/curated_borough_summary.parquet` (และ `.csv`) — ข้อมูลที่เชื่อมกับ `taxi_zone_lookup.csv` แยกรายเขต (Borough)
3. `data/curated/sample_curated_trips.parquet` (และ `.csv`) — ตัวอย่างข้อมูลระดับ Trip ที่แปลง Schema เป็นมาตรฐานเดียวกันและเชื่อมชื่อ Borough/Zone แล้ว

In [ ]:
curated_monthly = pd.read_parquet("data/curated/curated_monthly_summary.parquet")
sample_trips = pd.read_parquet("data/curated/sample_curated_trips.parquet")

print("📊 1) Curated Monthly Summary Dataset (First 8 rows):")
print(curated_monthly.head(8).to_string(index=False))
print("\n🚕 2) Sample Harmonized & Zone-Enriched Curated Trips (First 8 rows):")
print(sample_trips.head(8).to_string(index=False))

📊 1) Curated Monthly Summary Dataset (First 8 rows):
 partition_key year_month taxi_type  raw_rows  baseline_clean_rows  strict_clean_rows  strict_dropped_rows  retention_rate_pct  total_duration_hours  avg_duration_minutes  total_distance_miles  avg_distance_miles  total_fare_usd  avg_fare_usd top_pickup_borough
   fhv_2024-01    2024-01       fhv   1290116              1290116            1289685                  431             99.9666             514045.67                 23.91                   NaN                 NaN             NaN           NaN            Unknown
 fhvhv_2024-01    2024-01     fhvhv  19663930             19663739           19663739                  191             99.9990            6065446.53                 18.51           95144096.00                4.84    471133958.99         23.96          Manhattan
 green_2024-01    2024-01     green     56551                56369              56367                  184             99.6746              17119.82             

## 4. การวิเคราะห์ผลจาก Curated Output เพื่อตอบโจทย์หลัก

เราสามารถรันโมดูล `src.analyze` เพื่อดึงข้อมูลจาก `data/curated/` มาคำนวณหาประเภทแท็กซี่ที่ถูกใช้งานมากที่สุดในครึ่งแรกของปี 2024 ทั้งในมิติ **จำนวนเที่ยว (Total Clean Trips)** และ **เวลาให้บริการรวม (Total Passenger Ride Hours)** ได้ภายในเวลาไม่กี่วินาที:

In [ ]:
from src.analyze import run_analytics

final_summary_df = run_analytics()

🏆 FINAL ANALYTICAL RESULT (FROM CURATED DATASET)
 rank taxi_type  sub2_strict_clean_trips  trip_market_share_pct  sub1_baseline_clean_trips  raw_trips  dropped_anomalies  retention_rate_pct  total_duration_hours  duration_market_share_pct  avg_duration_minutes  total_distance_miles  avg_distance_miles  total_fare_usd  avg_fare_usd top_pickup_borough
    1     fhvhv                120853747                  80.93                  120853747  120864668              10921             99.9910           39945509.26                      81.30                 19.83          605943141.37                5.01    3.101489e+09         25.66          Manhattan
    2    yellow                 20015901                  13.40                   20016046   20332093             316192             98.4449            5643314.84                      11.49                 16.92           93495722.82                4.67    3.896061e+08         19.46          Manhattan
    3       fhv                  8117007  

## 5. การตัดสินใจสำคัญในการออกแบบ, เปรียบเทียบกับ Submission 1, ข้อจำกัด และ Learning Reflection

### 5.1 การตัดสินใจสำคัญในการออกแบบ Pipeline (Key Design Decisions)
1. **การแยกชั้นการจัดเก็บข้อมูล (Curated Storage Layer) และการทำ State Manifest (`outputs/pipeline_state.json`):**
   - แก้ปัญหาจาก Submission 1 ที่ต้อง `del df` ทิ้งหลังนับเสร็จใน RAM โดยบันทึกผลที่ทำความสะอาดแล้วลง `data/curated/` และเช็คสถานะจาก `pipeline_state.json` ทำให้การรันซ้ำ (Idempotent Re-run) ใช้เวลาเพียง **0.04 วินาที** (จากเดิม 30.52 วินาที) และวิเคราะห์ผลซ้ำจาก Curated Layer ได้ทันทีโดยไม่ต้องอ่านไฟล์ Raw 149.6 ล้านแถวใหม่
2. **การใช้ DuckDB Columnar Projection Pushdown & Vectorized SQL:**
   - ดึงเฉพาะคอลัมน์ที่จำเป็นจากไฟล์ Parquet ทั้ง 24 ไฟล์ (~3.12 GB) และเชื่อม (JOIN) กับตารางอ้างอิง `taxi_zone_lookup.csv` ได้อย่างรวดเร็วโดยไม่เกิดปัญหา RAM เต็ม
3. **การออกแบบ Data Quality Gate แบบ Dual-Tracking (`baseline_clean_rows` คู่กับ `strict_clean_rows`):**
   - เพิ่มกฎกรองวันที่นอกช่วงครึ่งแรกของปี 2024 (`2024-01-01` ถึง `2024-06-30`) และกรองเที่ยวที่ระยะเวลาเดินทางเกิน 24 ชั่วโมง (`> 1,440 นาที`) ซึ่งดักจับข้อมูลผิดปกติได้เพิ่มอีก **2,371 แถว** พร้อมทั้งเก็บค่า `baseline_clean_rows` (`149,327,659` แถว) ไว้พิสูจน์ความถูกต้องตรงกับ Submission 1 ได้ 100%

### 5.2 สรุปสั้น ๆ ว่า Submission 2 พัฒนาจาก Submission 1 อย่างไร และยังมีข้อจำกัดใดอยู่
Submission 2 ได้เปลี่ยนโค้ดวิเคราะห์แบบรันครั้งเดียวใน RAM ของ Submission 1 ให้กลายเป็น **Modular Data Pipeline** ที่เชื่อมข้อมูลทั้งจากไฟล์ Parquet 4 โครงสร้างและไฟล์อ้างอิง CSV (`taxi_zone_lookup.csv`), เพิ่มการตรวจคุณภาพข้อมูลที่รัดกุมขึ้นพร้อมออกรายงาน `data_quality_report.csv`, จัดเก็บผลลัพธ์ลง **Curated Layer** (`data/curated/`) และมีระบบบันทึกสถานะ (`pipeline_state.json`) ที่รองรับการรันซ้ำและอัปเดตข้อมูลใหม่ได้อย่างเป็นระบบ อย่างไรก็ตาม Pipeline ปัจจุบันยังมีข้อจำกัดจากข้อมูลต้นทางของกลุ่ม `fhv` ที่ไม่มีคอลัมน์ระยะทาง/ค่าโดยสารและไม่ระบุรหัสโซนขึ้นรถถึง ~78% รวมถึงข้อจำกัดขนาดไฟล์ของ GitHub (100 MB) ที่ทำให้เก็บเฉพาะ Curated Summary และ Sample Trips บน Repository แทนการเก็บข้อมูลระดับแถวครบทั้ง 149.3 ล้านแถว

### 5.3 Learning Reflection (3–5 ประโยค)
การพัฒนางานจาก Submission 1 มาเป็น Submission 2 ทำให้เข้าใจความแตกต่างระหว่างการเขียนสคริปต์วิเคราะห์ข้อมูลแบบครั้งเดียวจบ กับการออกแบบระบบ Data Pipeline ที่ใช้งานจริงในระยะยาว ในตอนทำ Submission 1 ปัญหาใหญ่ที่พบคือข้อจำกัดด้านRAM ที่บีบให้ต้องลบข้อมูลทิ้งหลังประมวลผลแต่ละไฟล์ ทำให้เมื่อต้องการปรับแก้เกณฑ์การทำความสะอาดหรือเพิ่มตัวแปรใหม่ๆ ต้องเสียเวลารันอ่านข้อมูลเกือบ 150 ล้านแถวใหม่ทั้งหมด การนำแนวคิดการแยกชั้นข้อมูล ร่วมกับระบบบันทึกสถานะการทำงาน และการใช้ DuckDB มาใช้ใน Submission 2 ช่วยแก้ปัญหาใน Submission 1 ได้ นอกจากนี้ การเพิ่มขั้นตอนตรวจสอบคุณภาพข้อมูล ที่แยกบันทึกสถิติความผิดปกติแต่ละสาเหตุอย่างชัดเจน ยังช่วยให้เราค้นพบข้อมูลที่หลุดขอบเขตเวลาปี 2024 และข้อมูลที่มิเตอร์ค้างเกิน 24 ชั่วโมง ซึ่งทำให้ผลลัพธ์สุดท้ายมีความน่าเชื่อถือและตรวจสอบย้อนกลับได้จริงในทุกขั้นตอน